In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
DATA_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
df_original = pd.read_csv(DATA_URL)

print("Dataset: Titanic Passenger Dataset")
print("Source:", DATA_URL)
print("File type: CSV")
print("Number of rows:", df_original.shape[0])
print("Number of columns:", df_original.shape[1])
print("Column names:", df_original.columns.tolist())

In [ ]:
display(df_original.head())
display(df_original.tail())
print("\\nDataset information:")
df_original.info()
print("\\nShape:", df_original.shape)
print("\\nData types:")
print(df_original.dtypes)
print("\\nStatistical summary:")
display(df_original.describe(include="all").T)

In [ ]:
numeric_cols = df_original.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df_original.select_dtypes(exclude=np.number).columns.tolist()

print("Numerical columns:", numeric_cols)
print("\\nCategorical columns:", categorical_cols)
print("\\nNumerical summary:")
display(df_original[numeric_cols].describe().T)

print("\\nPossible future target: survived")
print("Reason: it records whether a passenger survived. No model is trained in this activity.")

In [ ]:
missing_count = df_original.isna().sum()
missing_percent = (missing_count / len(df_original) * 100).round(2)
missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percentage": missing_percent
}).sort_values("Missing Percentage", ascending=False)

display(missing_summary)
missing_summary[missing_summary["Missing Count"] > 0].plot(
    kind="bar", y="Missing Percentage", legend=False, figsize=(10, 4),
    title="Percentage of Missing Values by Column"
)
plt.ylabel("Missing values (%)")
plt.xlabel("Column")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
df = df_original.copy()

# Remove columns that are unsuitable or redundant for this prepared version.
columns_to_drop = ["deck", "embark_town"]
df = df.drop(columns=columns_to_drop, errors="ignore")

# Median imputation for numerical age.
df["age"] = df["age"].fillna(df["age"].median())

# Mode imputation for categorical embarked.
if df["embarked"].isna().any():
    df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])

print("Missing values after handling:")
display(df.isna().sum().to_frame("Remaining missing values"))
print("Prepared shape:", df.shape)

In [ ]:
duplicate_count = df.duplicated().sum()
print("Exact duplicate rows before removal:", duplicate_count)

# Remove only exact duplicate rows from the working copy.
df = df.drop_duplicates().copy()

# Trim accidental spaces in text columns.
text_cols = df.select_dtypes(include="object").columns
for col in text_cols:
    df[col] = df[col].str.strip()

print("Exact duplicate rows after removal:", df.duplicated().sum())
print("Shape after duplicate check:", df.shape)

In [ ]:
print("Numerical summary:")
display(df.select_dtypes(include=np.number).describe().T)

print("\\nFrequency counts for categorical columns:")
for col in ["sex", "pclass", "embarked", "who"]:
    if col in df.columns:
        print(f"\\n{col}:")
        display(df[col].value_counts(dropna=False).to_frame("Count"))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=df, x="age", bins=25, kde=True, ax=axes[0])
axes[0].set_title("Distribution of Passenger Ages")
sns.countplot(data=df, x="pclass", hue="survived", ax=axes[1])
axes[1].set_title("Passenger Class vs Survival")
axes[1].set_xlabel("Passenger class")
axes[1].legend(title="Survived", labels=["No", "Yes"])
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="pclass", y="fare")
plt.title("Fare Distribution by Passenger Class")
plt.tight_layout()
plt.show()

In [ ]:
# Keep a copy for preprocessing demonstration.
ml_ready = df.copy()

# Remove duplicate/redundant representations and free-text labels.
redundant_cols = ["alive", "class", "who"]
ml_ready = ml_ready.drop(columns=redundant_cols, errors="ignore")

# One-hot encode nominal categorical columns.
categorical_for_encoding = ml_ready.select_dtypes(include=["object", "bool"]).columns.tolist()
ml_ready = pd.get_dummies(
    ml_ready,
    columns=categorical_for_encoding,
    drop_first=True,
    dtype=int
)

print("Categorical columns encoded:", categorical_for_encoding)
print("Shape before encoding (after redundant-column removal):", df.drop(columns=redundant_cols, errors="ignore").shape)
print("Shape after encoding:", ml_ready.shape)
display(ml_ready.head())

In [ ]:
from sklearn.preprocessing import StandardScaler

scaled_data = ml_ready.copy()
continuous_candidates = ["age", "fare", "sibsp", "parch"]
scale_cols = [c for c in continuous_candidates if c in scaled_data.columns]

scaler = StandardScaler()
scaled_data[scale_cols] = scaler.fit_transform(scaled_data[scale_cols])

print("Columns standardized:", scale_cols)
display(scaled_data[scale_cols].describe().T)
display(scaled_data.head())

In [ ]:
print("Final readable dataset shape:", df.shape)
print("Missing values in readable dataset:", int(df.isna().sum().sum()))
print("Duplicate rows in readable dataset:", int(df.duplicated().sum()))

print("\\nFinal encoded and scaled dataset shape:", scaled_data.shape)
print("Missing values in encoded/scaled dataset:", int(scaled_data.isna().sum().sum()))

df.to_csv("titanic_cleaned_readable.csv", index=False)
scaled_data.to_csv("titanic_prepared_encoded_scaled.csv", index=False)

print("\\nFiles saved:")
print("1. titanic_cleaned_readable.csv")
print("2. titanic_prepared_encoded_scaled.csv")
display(df.head())
display(scaled_data.head())